# IR – Work Session 2: Term Weighting
Master 2 – Computer Science

TF, normalized TF, DF, IDF-1, IDF-2 and TF-IDF implemented from scratch, then verified with scikit-learn.

## 0. Corpus and conventions

In [1]:
import math
import os
from collections import Counter

import pandas as pd

pd.set_option("display.precision", 4)
os.makedirs("results", exist_ok=True)

corpus = {
    "D1": "information retrieval retrieval system",
    "D2": "information system system system",
    "D3": "retrieval model system",
    "D4": "information retrieval model model",
}
VOCABULARY = ["information", "retrieval", "system", "model"]  # fixed order
documents = {name: text.split() for name, text in corpus.items()}
N = len(documents)
documents

{'D1': ['information', 'retrieval', 'retrieval', 'system'],
 'D2': ['information', 'system', 'system', 'system'],
 'D3': ['retrieval', 'model', 'system'],
 'D4': ['information', 'retrieval', 'model', 'model']}

## Part 1 – Term Frequency

In [2]:
def compute_tf(document):
    """Raw TF of a tokenized document.

    Input : list of tokens
    Output: dict {term: number of occurrences in the document}
    """
    return dict(Counter(document))


def compute_normalized_tf(document):
    """Normalized TF = tf(t,d) / total number of terms in d.

    Input : list of tokens
    Output: dict {term: tf / len(document)}
    """
    total = len(document)
    return {term: count / total for term, count in compute_tf(document).items()}


def build_matrix(per_doc_values, vocabulary=VOCABULARY):
    """Rows = documents, columns = vocabulary in the FIXED order.
    Missing terms get 0."""
    rows = {doc: [values.get(t, 0) for t in vocabulary] for doc, values in per_doc_values.items()}
    return pd.DataFrame.from_dict(rows, orient="index", columns=vocabulary)


tf_matrix = build_matrix({d: compute_tf(toks) for d, toks in documents.items()})
ntf_matrix = build_matrix({d: compute_normalized_tf(toks) for d, toks in documents.items()})
print("Raw TF matrix"); display(tf_matrix)
print("Normalized TF matrix"); display(ntf_matrix)

Raw TF matrix


,information,retrieval,system,model
D1,1,2,1,0
D2,1,0,3,0
D3,0,1,1,1
D4,1,1,0,2


Normalized TF matrix


,information,retrieval,system,model
D1,0.25,0.5000,0.2500,0.0000
D2,0.25,0.0000,0.7500,0.0000
D3,0.00,0.3333,0.3333,0.3333
D4,0.25,0.2500,0.0000,0.5000


In [3]:
def most_frequent_terms(tf_row):
    """All terms reaching the maximum TF (ties are all reported)."""
    max_tf = tf_row.max()
    return list(tf_row[tf_row == max_tf].index), max_tf


for doc, row in tf_matrix.iterrows():
    terms, value = most_frequent_terms(row)
    print(f"{doc}: {terms} (tf = {value})")

D1: ['retrieval'] (tf = 2)
D2: ['system'] (tf = 3)
D3: ['retrieval', 'system', 'model'] (tf = 1)
D4: ['model'] (tf = 2)


**Question 4 – result.** D1: `retrieval` (2) · D2: `system` (3) · D3: tie between `retrieval`, `system` and `model` (1 each) · D4: `model` (2).

## Part 2 – Document Frequency and IDF

In [4]:
def compute_df(documents, vocabulary=VOCABULARY):
    """df(t) = number of documents containing t at least once.
    A term counts at most once per document (we use set())."""
    return {t: sum(1 for tokens in documents.values() if t in set(tokens)) for t in vocabulary}


def compute_idf(df, n_documents):
    """IDF-1: idf(t) = log10(N / df(t))."""
    return {t: math.log10(n_documents / d) for t, d in df.items()}


def compute_idf_smooth(df, n_documents):
    """IDF-2: idf_smooth(t) = log10(1 + N / df(t))."""
    return {t: math.log10(1 + n_documents / d) for t, d in df.items()}


df = compute_df(documents)
idf1 = compute_idf(df, N)
idf2 = compute_idf_smooth(df, N)

df_idf = pd.DataFrame({"DF": df, "IDF-1": idf1, "IDF-2": idf2}).loc[VOCABULARY]
df_idf.index.name = "term"
df_idf

,DF,IDF-1,IDF-2
term,,,
information,3,0.1249,0.3680
retrieval,3,0.1249,0.3680
system,3,0.1249,0.3680
model,2,0.3010,0.4771


In [5]:
for col in ["IDF-1", "IDF-2"]:
    print(f"{col}: highest = {df_idf[col].idxmax()} ({df_idf[col].max():.4f}), "
          f"lowest = {df_idf[col].idxmin()} ({df_idf[col].min():.4f}) "
          f"[ties for lowest: {list(df_idf.index[df_idf[col] == df_idf[col].min()])}]")

IDF-1: highest = model (0.3010), lowest = information (0.1249) [ties for lowest: ['information', 'retrieval', 'system']]
IDF-2: highest = model (0.4771), lowest = information (0.3680) [ties for lowest: ['information', 'retrieval', 'system']]


**Question 8 – result.** Under both conventions the highest IDF is `model` (DF = 2: IDF-1 = log10(4/2) = 0.3010, IDF-2 = log10(3) = 0.4771). The lowest IDF is shared by `information`, `retrieval` and `system` (DF = 3: IDF-1 = log10(4/3) = 0.1249, IDF-2 = log10(7/3) = 0.3680).

**Question 9 – a term in every document.** Then df = N, so IDF-1 = log10(N/N) = log10(1) = **0**: the term carries no discriminative power and its TF-IDF weight is 0 whatever its TF. In this corpus no term appears in all four documents, but the three terms with DF = 3 get a very low IDF-1 (0.1249). IDF-2 never reaches 0: for df = N it gives log10(2) = 0.3010, so frequent terms keep a positive weight. IDF-2 also compresses the gap between rare and common terms (0.4771 vs 0.3680 instead of 0.3010 vs 0.1249) while preserving the order of terms.

## Part 3 – TF-IDF weighting

In [6]:
def compute_tfidf(tf_values, idf_values):
    """w(t,d) = tf(t,d) x idf(t).

    Input : tf_values  -> dict {term: tf} for ONE document
            idf_values -> dict {term: idf}
    Output: dict {term: weight} (terms absent from the document are omitted)
    """
    return {t: tf * idf_values[t] for t, tf in tf_values.items()}


tfidf_raw = build_matrix({d: compute_tfidf(compute_tf(toks), idf1) for d, toks in documents.items()})
tfidf_norm = build_matrix({d: compute_tfidf(compute_normalized_tf(toks), idf1) for d, toks in documents.items()})

print("TF-IDF (raw TF x IDF-1)"); display(tfidf_raw)
print("TF-IDF (normalized TF x IDF-1)"); display(tfidf_norm)

TF-IDF (raw TF x IDF-1)


,information,retrieval,system,model
D1,0.1249,0.2499,0.1249,0.0000
D2,0.1249,0.0000,0.3748,0.0000
D3,0.0000,0.1249,0.1249,0.3010
D4,0.1249,0.1249,0.0000,0.6021


TF-IDF (normalized TF x IDF-1)


,information,retrieval,system,model
D1,0.0312,0.0625,0.0312,0.0000
D2,0.0312,0.0000,0.0937,0.0000
D3,0.0000,0.0416,0.0416,0.1003
D4,0.0312,0.0312,0.0000,0.1505


In [7]:
def top_terms(matrix):
    """Highest-weighted term(s) per document (ties reported)."""
    out = {}
    for doc, row in matrix.iterrows():
        m = row.max()
        out[doc] = [t for t in row.index if math.isclose(row[t], m)]
    return out


print("Raw      :", top_terms(tfidf_raw))
print("Normalized:", top_terms(tfidf_norm))

Raw      : {'D1': ['retrieval'], 'D2': ['system'], 'D3': ['model'], 'D4': ['model']}
Normalized: {'D1': ['retrieval'], 'D2': ['system'], 'D3': ['model'], 'D4': ['model']}


**Question 13 – comparison.** The highest-weighted term is the same under both versions: D1 → `retrieval`, D2 → `system`, D3 → `model`, D4 → `model`. The ranking inside each document does not change because dividing by the document length multiplies the whole row by the same constant. What changes is the **scale** and the **comparison across documents**: with raw TF, D2's `system` (0.3748) beats D1's `retrieval` (0.2499); with normalized TF the values are on a common length-relative scale (D3 is 3 terms long, the others 4). Note D3: every term has TF = 1, so the ranking is decided only by IDF, and `model` (the rarest term) wins.

**Question 14 – high TF ≠ high TF-IDF.** TF-IDF multiplies TF by IDF. A term can be frequent in a document but also frequent in the collection, so its IDF is low and the product stays small. Example: in D2, `system` has TF = 3 but IDF-1 = 0.1249 → w = 0.3748, while in D4 `model` has TF = 2 but IDF-1 = 0.3010 → w = 0.6021, which is higher despite the lower TF.

## Part 4 – Verification with scikit-learn

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    vocabulary=VOCABULARY,   # same fixed order
    sublinear_tf=False,
    smooth_idf=False,
    norm=None,
)
sk_matrix = vectorizer.fit_transform(corpus.values())
sk_tfidf = pd.DataFrame(sk_matrix.toarray(), index=corpus.keys(), columns=VOCABULARY)
print("scikit-learn TF-IDF (smooth_idf=False, norm=None)"); display(sk_tfidf)

sk_idf = pd.Series(vectorizer.idf_, index=VOCABULARY)
comparison = pd.DataFrame({
    "my IDF-1 (log10)": df_idf["IDF-1"],
    "sklearn idf": sk_idf,
    "ln(N/df)": [math.log(N / df[t]) for t in VOCABULARY],
    "sklearn idf - 1": sk_idf - 1,
})
comparison

scikit-learn TF-IDF (smooth_idf=False, norm=None)


,information,retrieval,system,model
D1,1.2877,2.5754,1.2877,0.0000
D2,1.2877,0.0000,3.8630,0.0000
D3,0.0000,1.2877,1.2877,1.6931
D4,1.2877,1.2877,0.0000,3.3863


,my IDF-1 (log10),sklearn idf,ln(N/df),sklearn idf - 1
information,0.1249,1.2877,0.2877,0.2877
retrieval,0.1249,1.2877,0.2877,0.2877
system,0.1249,1.2877,0.2877,0.2877
model,0.3010,1.6931,0.6931,0.6931


In [9]:
# Step 17: isolate the source of the difference.
# 1) raw counts are identical (sklearn tf = our tf):
counts = pd.DataFrame(
    __import__("sklearn.feature_extraction.text", fromlist=["CountVectorizer"])
    .CountVectorizer(vocabulary=VOCABULARY).fit_transform(corpus.values()).toarray(),
    index=corpus.keys(), columns=VOCABULARY,
)
print("Raw counts identical:", counts.equals(tf_matrix.astype(int)))

# 2) sklearn idf = ln(N/df) + 1 when smooth_idf=False
print("sklearn idf == ln(N/df) + 1 :", all(
    math.isclose(sk_idf[t], math.log(N / df[t]) + 1) for t in VOCABULARY))

# 3) log10 and ln only differ by a constant: ln(x) = ln(10) * log10(x)
print("ln(N/df) == ln(10) * IDF-1  :", all(
    math.isclose(math.log(N / df[t]), math.log(10) * idf1[t]) for t in VOCABULARY))

# 4) Rebuild our matrix from sklearn's output: tf * (sk_idf - 1) / ln(10)
rebuilt = tf_matrix.mul(sk_idf - 1, axis=1) / math.log(10)
print("tf*(sk_idf-1)/ln(10) == my TF-IDF:", ((rebuilt - tfidf_raw).abs().max().max()) < 1e-12)

# Verified result: D2 / system
print(f"D2/system  mine = {tfidf_raw.loc['D2','system']:.4f}   sklearn = {sk_tfidf.loc['D2','system']:.4f}")

Raw counts identical: True
sklearn idf == ln(N/df) + 1 : True
ln(N/df) == ln(10) * IDF-1  : True
tf*(sk_idf-1)/ln(10) == my TF-IDF: True
D2/system  mine = 0.3748   sklearn = 3.8630


**Questions 16–18 – interpretation.** scikit-learn with `smooth_idf=False` uses `idf(t) = ln(N / df(t)) + 1`, whereas the course uses `idf(t) = log10(N / df(t))`. Two differences explain every gap:

1. **Logarithm base:** natural log vs. base 10 (`ln x = ln(10) · log10 x ≈ 2.3026 · log10 x`).
2. **The `+1` term:** scikit-learn adds 1 to the IDF, so even a term present in every document keeps a weight (IDF = 1) instead of 0.

The raw counts (TF) are identical to ours. **Verified result:** D2 / `system` → ours = 3 × 0.1249 = 0.3748; scikit-learn = 3 × (ln(4/3) + 1) = 3 × 1.2877 = 3.8630. Removing the `+1` and converting ln → log10 gives exactly 0.3748. Normalization is not a source of difference here because we set `norm=None` (by default scikit-learn would L2-normalize each row); `smooth_idf=True` would additionally change df and N to `ln((1+N)/(1+df)) + 1`.

In [10]:
# Save all deliverables
tf_matrix.to_csv("results/tf_matrix.csv")
ntf_matrix.to_csv("results/normalized_tf_matrix.csv")
df_idf.to_csv("results/df_idf_comparison.csv")
tfidf_raw.to_csv("results/tfidf_raw_matrix.csv")
tfidf_norm.to_csv("results/tfidf_normalized_matrix.csv")
sorted(os.listdir("results"))

['df_idf_comparison.csv',
 'normalized_tf_matrix.csv',
 'tf_matrix.csv',
 'tfidf_normalized_matrix.csv',
 'tfidf_raw_matrix.csv']

## Analysis questions

**19. Does high TF always give a high TF-IDF?** No. w = tf × idf: if idf is close to 0 (term present in almost all documents), a large tf is cancelled. In D2, `system` has tf = 3 but idf-1 = 0.1249 (w = 0.3748), whereas `model` in D4 has tf = 2 and idf-1 = 0.3010 (w = 0.6021).

**20. Effect of normalized TF.** It divides counts by document length, so weights become relative frequencies and are comparable between short and long documents (D3 has 3 terms, the others 4). A long document no longer gets large weights just because it repeats more words. It does not change the ranking inside a document, only the scale.

**21. A term in every document.** IDF-1 = log10(N/N) = 0, so the term gets weight 0 in all documents. IDF-2 = log10(1+1) = log10(2) ≈ 0.301 > 0: the term is down-weighted but never ignored completely.

**22. High frequency, low IDF.** `system` is frequent in D2 (tf = 3) and appears in 3 of 4 documents (idf-1 = 0.1249). Its high TF is largely cancelled by the low IDF; the typical case in real corpora is stop-words ("the", "of"), which get an IDF close to 0.

**23. Why keep the vocabulary order?** Each column of the matrix corresponds to one term. If the order changes between matrices (or between a document and a query), the same column index would represent different terms, so vector products, similarities and comparisons (e.g. with scikit-learn's output) would be silently wrong.